# 01 · Country explorer — reading the shape of a country's internet

One row per country per month: the broadest view, and the right place to start.

**What you will learn**
- how to read a percentile curve — the *shape* of a country's quality;
- why the median is the headline number, and why the mean does not exist here;
- how far the top of a country's distribution stretches past its middle;
- whether a ranking survives when you move from p50 to p95.

**Start with one worked question:** *Is the US median download faster than Germany's?* Run the two cells below, then the widgets are yours — ask your own question about your own country.

## Setup — load the data the way notebook 00 taught you

In [ ]:
# ── Setup: imports, manifest, catalog ─────────────────────────────────────────
# The same three-step pattern is used by every notebook in this folder:
#   1. read the manifest (a list of every published file)
#   2. find the URL for the month/slice you want
#   3. pd.read_parquet(url) — done.

import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

MANIFEST_URL = "https://measurementlab.net/data/stats/manifest.json"
manifest = requests.get(MANIFEST_URL, timeout=30).json()

records = []
for path, meta in manifest["files"].items():
    parts = path.split("/")
    # cache/v1/{start_ts}/{end_ts}/{slice_name}/data.parquet
    if len(parts) == 6 and parts[5] == "data.parquet":
        records.append({
            "start": pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "end":   pd.to_datetime(parts[3], format="%Y%m%dT%H%M%SZ"),
            "slice": parts[4],
            "url":   meta["url"],
        })

catalog = (pd.DataFrame(records)
           .sort_values(["slice", "start"])
           .reset_index(drop=True))


def month_url(slice_name, start):
    # Direct URL of one month of one slice. start: 'YYYY-MM-DD' (first of month).
    row = catalog[(catalog["slice"] == slice_name) &
                  (catalog["start"] == pd.to_datetime(start))]
    if row.empty:
        raise ValueError(f"No {slice_name} file for {start}")
    return row.iloc[0]["url"]


def latest_month(slice_name):
    # Newest month that exists for a slice, as 'YYYY-MM-DD'.
    return catalog.loc[catalog["slice"] == slice_name, "start"].max().strftime("%Y-%m-%d")

print("Catalog loaded —", len(catalog), "files,",
      catalog["slice"].nunique(), "slices,",
      catalog["start"].min().date(), "→", catalog["start"].max().date())

### The worked question

In [ ]:
MONTH = latest_month("downloads_by_country")
df = pd.read_parquet(month_url("downloads_by_country", MONTH))

for cc in ["US", "DE"]:
    val = df[df["country_code"] == cc]["download_p50"].iloc[0]
    tests = df[df["country_code"] == cc]["sample_count"].iloc[0]
    print(f"{cc}: median download {val:.0f} Mbit/s  ({tests:,} tests, {MONTH})")

## The four metrics, live

Pick a month, a metric, and how many countries to show. Every chart is labelled **Highest N**, not "Top N": this is a position on a scale, not a prize. The caption under every chart is a plain-language reading of what you are looking at — say it aloud, in your own words.

In [ ]:
METRICS = {
    "Download (Mbit/s)":  ("download_p50", "downloads_by_country", False),
    "Upload (Mbit/s)":    ("upload_p50",   "uploads_by_country",   False),
    "Latency (ms)":       ("latency_p50",  "downloads_by_country", True),
    "Packet loss":        ("loss_p50",     "downloads_by_country", True),
}

country_months = sorted(
    catalog.loc[catalog["slice"] == "downloads_by_country", "start"]
    .dt.strftime("%Y-%m-%d").unique(), reverse=True)

w_month = widgets.Dropdown(options=country_months, value=MONTH,
                            description="Month:", layout=widgets.Layout(width="230px"))
w_metric = widgets.Dropdown(options=list(METRICS), value="Download (Mbit/s)",
                             description="Metric:", layout=widgets.Layout(width="250px"))
w_n = widgets.IntSlider(value=20, min=5, max=60, step=5,
                         description="Highest N:", layout=widgets.Layout(width="360px"))
w_min = widgets.IntSlider(value=1000, min=0, max=100000, step=100,
                           description="Min tests:", layout=widgets.Layout(width="360px"))
out = widgets.Output()


def update(change=None):
    col, slice_name, lower = METRICS[w_metric.value]
    data = pd.read_parquet(month_url(slice_name, w_month.value))
    data = data[data["sample_count"] >= w_min.value]
    if data.empty:
        with out:
            clear_output(wait=True)
            print("No countries pass the minimum test count — lower 'Min tests'.")
        return
    top = data.nsmallest(w_n.value, col) if lower else data.nlargest(w_n.value, col)
    top = top.sort_values(col, ascending=not lower)
    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, max(5, w_n.value * 0.36)))
        ax.barh(top["country_code"], top[col])
        ax.set_xlabel(w_metric.value)
        ax.set_title(f"Highest {w_n.value} countries by {w_metric.value} — {w_month.value}")
        plt.tight_layout(); plt.show()
        first = top.iloc[0]
        print(f"Reading it aloud: {first['country_code']} has the highest "
              f"{w_metric.value.lower()} ({first[col]:.0f}) this month, "
              f"from {first['sample_count']:,} tests.")


for w in (w_month, w_metric, w_n, w_min):
    w.observe(update, "value")
display(widgets.VBox([widgets.HBox([w_month, w_metric]), w_n, w_min, out]))
update()

## The shape — percentile curves

The medians above are one point each. A country is not a point: it is a *curve*. Plotting all nine percentiles (p1 → p99) draws the shape of the distribution.

**How to read the curve** — it is the whole distribution drawn percentile by percentile:
- the **left edge is the slowest ~1%**, the **middle is the median**, the **right edge is the fastest ~1%**;
- **steep** means few connections spread over a wide speed range — most countries show this long tail of very fast connections near p90–p99;
- **flat** means many connections packed into a narrow band — here the bulk of users actually live;
- **lower on the curve is not "worse off"** — it just sits where the mass of the distribution is. Shape is descriptive: a short tight shape means most people are similar; a stretched top means a minority far out front. Neither alone is a judgement.

For latency and loss the curve slopes the *other* way (low percentile = high latency) — the polarity flip from notebook 00, visible.

In [ ]:
PC_METRICS = {
    "Download (Mbit/s)": ("download", "downloads_by_country"),
    "Upload (Mbit/s)":   ("upload",   "uploads_by_country"),
    "Latency (ms)":      ("latency",  "downloads_by_country"),
    "Packet loss":       ("loss",     "downloads_by_country"),
}

w_pm = widgets.Dropdown(options=country_months, value=MONTH, description="Month:",
                         layout=widgets.Layout(width="220px"))
w_pmetric = widgets.Dropdown(options=list(PC_METRICS), value="Download (Mbit/s)",
                              description="Metric:", layout=widgets.Layout(width="250px"))
w_pcountries = widgets.SelectMultiple(
    options=sorted(df[df["sample_count"] >= 1000]["country_code"].tolist()),
    value=["US", "DE", "BR", "IN", "NG"], description="Countries:", rows=8,
    layout=widgets.Layout(width="180px"))
out_p = widgets.Output()


def update_p(change=None):
    prefix, slice_name = PC_METRICS[w_pmetric.value]
    data = pd.read_parquet(month_url(slice_name, w_pm.value))
    pcols = sorted([c for c in data.columns if c.startswith(f"{prefix}_p")],
                   key=lambda c: int(c.split("_p")[1]))
    pnums = [int(c.split("_p")[1]) for c in pcols]
    with out_p:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))
        for cc in w_pcountries.value:
            row = data[data["country_code"] == cc]
            if row.empty:
                continue
            ax.plot(pnums, [float(row.iloc[0][c]) for c in pcols],
                    marker="o", markersize=4, label=cc)
        ax.set_xlabel("Percentile")
        ax.set_ylabel(w_pmetric.value)
        ax.set_title(f"Percentile curve — {w_pmetric.value} — {w_pm.value}")
        ax.legend(title="Country"); plt.tight_layout(); plt.show()


w_pm.observe(update_p, "value"); w_pmetric.observe(update_p, "value")
w_pcountries.observe(update_p, "value")
display(widgets.VBox([widgets.HBox([w_pm, w_pmetric]), w_pcountries, out_p]))
update_p()

## The p95/p50 shape gauge

One number that summarises the shape: **p95 ÷ p50**. A ratio near 1 means the top of the distribution sits close to its middle (a tight shape). A ratio of 3–4 means the top stretches 3–4× past the median — a long tail.

This is a *shape* gauge, not an index of anything. Skew is descriptive: "stretched to one side". The same ratio can come from very different underlying populations.

In [ ]:
w_gm = widgets.Dropdown(options=country_months, value=MONTH, description="Month:",
                         layout=widgets.Layout(width="220px"))
w_gmin = widgets.IntSlider(value=1000, min=0, max=100000, step=100,
                            description="Min tests:", layout=widgets.Layout(width="360px"))
out_g = widgets.Output()


def update_g(change=None):
    data = pd.read_parquet(month_url("downloads_by_country", w_gm.value))
    data = data[data["sample_count"] >= w_gmin.value].copy()
    data["gauge"] = data["download_p95"] / data["download_p50"]
    lo = data.nsmallest(10, "gauge")
    hi = data.nlargest(10, "gauge")
    with out_g:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        for ax, part, title in ((axes[0], hi, "Most stretched (longest tail)"),
                                (axes[1], lo, "Tightest (top close to middle)")):
            part = part.sort_values("gauge")
            ax.barh(part["country_code"], part["gauge"])
            ax.set_xlabel("p95 / p50 ratio")
            ax.set_title(f"{title} — {w_gm.value}")
        plt.tight_layout(); plt.show()
        worst = hi.iloc[0]
        print(f"Reading it aloud: {worst['country_code']}'s top 5% of connections run "
              f"{worst['gauge']:.1f}x the speed of its median user — the shape is "
              f"stretched, from {worst['sample_count']:,} tests.")


w_gm.observe(update_g, "value"); w_gmin.observe(update_g, "value")
display(widgets.VBox([widgets.HBox([w_gm, w_gmin]), out_g]))
update_g()

## Percentile sweep — do the rankings hold?

Which number should you trust, p50 or p95? Both — but they answer different questions. Drag the percentile slider and watch the ranking **re-order live**. Rankings that survive the whole sweep are robust facts; rankings that flip at p95 are the shape talking.

In [ ]:
SWEEP_METRICS = {
    "Download": ("download", "downloads_by_country", False),
    "Upload":   ("upload",   "uploads_by_country",   False),
    "Latency":  ("latency",  "downloads_by_country", True),
    "Loss":     ("loss",     "downloads_by_country", True),
}

w_sm = widgets.Dropdown(options=country_months, value=MONTH, description="Month:",
                         layout=widgets.Layout(width="220px"))
w_smetric = widgets.Dropdown(options=list(SWEEP_METRICS), value="Download",
                              description="Metric:", layout=widgets.Layout(width="220px"))
w_pct = widgets.IntSlider(value=50, min=5, max=99, step=5, description="Percentile:",
                           layout=widgets.Layout(width="360px"))
w_sn = widgets.IntSlider(value=15, min=5, max=40, step=5, description="Highest N:",
                          layout=widgets.Layout(width="360px"))
w_smin = widgets.IntSlider(value=1000, min=0, max=100000, step=100,
                            description="Min tests:", layout=widgets.Layout(width="360px"))
out_s = widgets.Output()


def update_s(change=None):
    prefix, slice_name, lower = SWEEP_METRICS[w_smetric.value]
    data = pd.read_parquet(month_url(slice_name, w_sm.value))
    data = data[data["sample_count"] >= w_smin.value]
    col = f"{prefix}_p{w_pct.value}"
    if col not in data.columns or data.empty:
        with out_s:
            clear_output(wait=True)
            print("No data for that percentile and minimum — adjust the controls.")
        return
    top = data.nsmallest(w_sn.value, col) if lower else data.nlargest(w_sn.value, col)
    top = top.sort_values(col, ascending=not lower)
    with out_s:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, max(5, w_sn.value * 0.36)))
        ax.barh(top["country_code"], top[col])
        ax.set_xlabel(f"p{w_pct.value} (Mbit/s)" if prefix in ("download", "upload")
                      else f"p{w_pct.value} ({'ms' if prefix == 'latency' else 'rate'})")
        ax.set_title(f"p{w_pct.value} — highest {w_sn.value} countries by {w_smetric.value.lower()}"
                     f" {'(lowest is best)' if lower else ''} — {w_sm.value}")
        plt.tight_layout(); plt.show()
        first = top.iloc[0]
        print(f"Reading it aloud: at p{w_pct.value}, {first['country_code']} leads "
              f"({first[col]:.0f}). Adjust the percentile and watch the order move.")


for w in (w_sm, w_smetric, w_pct, w_sn, w_smin):
    w.observe(update_s, "value")
display(widgets.VBox([widgets.HBox([w_sm, w_smetric]), w_pct, w_sn, w_smin, out_s]))
update_s()

## Check yourself

**Question.** Why is `latency_p95` the *fastest* 5% of connections, not the slowest?

**Answer.** For latency and loss, lower is better, so the dataset flips the percentiles: a higher percentile number always means a *better* connection. `latency_p95` is the 5% of tests with the lowest (best) latency. Scroll back to the latency curve above and check the direction it slopes — the polarity flip, visible. Next: **02 · the splits**.